# 🌿 VanaVaidhya — Plant Classifier Training
**EfficientNetV2S | 3-Phase | MixUp | Label Smoothing | TTA**

Target: **90%+ accuracy** on 71 Indian Medicinal Plant species

| Phase | Strategy | Expected Accuracy |
|-------|----------|------------------|
| Phase 1 (30 ep) | Head only, base frozen | 50–65% |
| Phase 2 (30 ep) | Top-100 conv + MixUp | 80–87% |
| Phase 3 (20 ep) | All conv, tiny LR | **88–93%** |

> **Before running:** Upload `medicinal_leaves.zip` to Google Drive root

In [1]:
# CELL 1 — Check GPU
import subprocess
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True)
print("GPU:", result.stdout.strip())

import tensorflow as tf
print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

GPU: Tesla T4, 15360 MiB
TF version: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted at /content/drive/MyDrive/")

Mounted at /content/drive
Drive mounted at /content/drive/MyDrive/


In [9]:
import os, zipfile, glob

# ⚙️  UPDATE THIS if you saved the zip in a subfolder
DRIVE_ZIP  = "/content/drive/MyDrive/medicinal_leaves.zip"
EXTRACT_TO = "/content/dataset"

os.makedirs(EXTRACT_TO, exist_ok=True)
print("Unzipping dataset (this takes 2–5 minutes)...")
with zipfile.ZipFile(DRIVE_ZIP, "r") as z:
    z.extractall(EXTRACT_TO)
print("Done!")

all_imgs = (glob.glob(f"{EXTRACT_TO}/**/*.jpg", recursive=True) +
            glob.glob(f"{EXTRACT_TO}/**/*.png", recursive=True))
print(f"Found {len(all_imgs)} images")

# Corrected paths
LEAF_DS  = f"{EXTRACT_TO}/Indian Medicinal Leaves Image Datasets/Medicinal Leaf dataset"
PLANT_DS = f"{EXTRACT_TO}/Indian Medicinal Leaves Image Datasets/Medicinal plant dataset"
print("Leaf dataset exists:",  os.path.exists(LEAF_DS))
print("Plant dataset exists:", os.path.exists(PLANT_DS))

Unzipping dataset (this takes 2–5 minutes)...
Done!
Found 12845 images
Leaf dataset exists: True
Plant dataset exists: True


In [4]:
# CELL 4 — Verify packages (all pre-installed on Colab)
import numpy as np
from PIL import Image
import sklearn, matplotlib
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("PIL:", Image.__version__)
print("✅ All packages ready")

numpy: 2.0.2
scikit-learn: 1.6.1
PIL: 11.3.0
✅ All packages ready


In [5]:
# CELL 5 — Class Mapping (inlined — no uploads needed)
LEAF_DATASET_MAP = {
    "Aloevera": "aloe_vera", "Amla": "amla", "Amruthaballi": "amruthaballi",
    "Arali": "arali", "Ashoka": "ashoka", "ashoka": "ashoka",
    "Astma_weed": "asthma_weed", "Badipala": "badipala",
    "Balloon_Vine": "balloon_vine", "Bamboo": "bamboo",
    "Beans": "SKIP", "Betel": "betel", "Bhrami": "brahmi",
    "Bringaraja": "bringaraja", "camphor": "camphor",
    "Caricature": "caricature", "Castor": "castor",
    "Catharanthus": "catharanthus", "Chakte": "chakte",
    "Chilly": "SKIP", "Citron lime (herelikai)": "citron_lime",
    "Coffee": "SKIP", "Common rue(naagdalli)": "common_rue",
    "Coriender": "coriander", "Curry": "curry_leaf",
    "Doddpathre": "doddpathre", "Drumstick": "moringa",
    "Ekka": "ekka", "Eucalyptus": "eucalyptus",
    "Ganigale": "ganigale", "Ganike": "ganike", "Gasagase": "gasagase",
    "Ginger": "ginger", "Globe Amarnath": "globe_amaranth",
    "Guava": "guava", "Henna": "henna", "Hibiscus": "hibiscus",
    "Honge": "honge", "Insulin": "insulin_plant",
    "Jackfruit": "jackfruit", "Jasmine": "jasmine",
    "kamakasturi": "kamakasturi", "Kambajala": "kambajala",
    "Kasambruga": "kasambruga", "kepala": "kepala",
    "Kohlrabi": "SKIP", "Lantana": "lantana",
    "Lemon": "lemon", "Lemongrass": "lemongrass",
    "Malabar_Nut": "malabar_nut", "Malabar_Spinach": "SKIP",
    "Mango": "mango", "Marigold": "marigold", "Mint": "mint",
    "Neem": "neem", "Nelavembu": "nelavembu", "Nerale": "nerale",
    "Nooni": "noni", "Onion": "SKIP", "Padri": "padri",
    "Palak(Spinach)": "SKIP", "Papaya": "papaya",
    "Parijatha": "harsingar", "Pea": "SKIP", "Pepper": "pepper",
    "Pomoegranate": "pomegranate", "Pumpkin": "SKIP",
    "Raddish": "SKIP", "Rose": "rose", "Sampige": "sampige",
    "Sapota": "sapota", "Seethaashoka": "ashoka",
    "Seethapala": "custard_apple", "Spinach1": "SKIP",
    "Tamarind": "tamarind", "Taro": "SKIP", "Tecoma": "tecoma",
    "Thumbe": "thumbe", "Tomato": "SKIP", "Tulsi": "tulsi",
    "Turmeric": "turmeric",
}
PLANT_DATASET_MAP = {
    "Aloevera": "aloe_vera", "Amla": "amla",
    "Amruta_Balli": "amruthaballi", "Arali": "arali",
    "Ashoka": "ashoka", "Ashwagandha": "ashwagandha",
    "Avacado": "SKIP", "Bamboo": "bamboo", "Basale": "SKIP",
    "Betel": "betel", "Betel_Nut": "betel_nut", "Brahmi": "brahmi",
    "Castor": "castor", "Curry_Leaf": "curry_leaf",
    "Doddapatre": "doddpathre", "Ekka": "ekka", "Ganike": "ganike",
    "Gauva": "guava", "Geranium": "geranium", "Henna": "henna",
    "Hibiscus": "hibiscus", "Honge": "honge",
    "Insulin": "insulin_plant", "Jasmine": "jasmine",
    "Lemon": "lemon", "Lemon_grass": "lemongrass", "Mango": "mango",
    "Mint": "mint", "Nagadali": "common_rue", "Neem": "neem",
    "Nithyapushpa": "catharanthus", "Nooni": "noni",
    "Pappaya": "papaya", "Pepper": "pepper",
    "Pomegranate": "pomegranate", "Raktachandini": "raktachandini",
    "Rose": "rose", "Sapota": "sapota", "Tulasi": "tulsi",
    "Wood_sorel": "wood_sorel",
}
TOXIC_PLANTS   = {"lantana", "arali", "makoy"}
CAUTION_PLANTS = {"castor", "catharanthus", "common_rue"}
AYUSH_PLANTS   = {
    "tulsi", "ashwagandha", "turmeric", "neem", "ginger", "amla",
    "brahmi", "bael", "harsingar", "moringa", "aloe_vera", "mint",
    "guava", "lemon", "hibiscus", "curry_leaf", "pepper", "coriander",
    "lemongrass", "jasmine", "pomegranate", "papaya", "amruthaballi",
    "nelavembu", "bringaraja", "doddpathre", "malabar_nut", "camphor",
    "tamarind", "nerale", "henna", "marigold", "castor",
}
print("✅ Class mappings loaded")

✅ Class mappings loaded


In [6]:
# CELL 6 — GPU Config & Hyperparameters
import tensorflow as tf, os

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print(f"✅ GPU ready: {gpus[0].name}")
    print("✅ Mixed precision float16 enabled")
    BATCH_SIZE = 128   # T4/V100=128, A100=256
else:
    BATCH_SIZE = 32
    print("⚠️  No GPU — change runtime type to T4 GPU!")

IMG_SIZE        = 260
PHASE1_EPOCHS   = 30
PHASE2_EPOCHS   = 30
PHASE3_EPOCHS   = 20
LR1             = 3e-4
LR2             = 5e-5
LR3             = 5e-6
LABEL_SMOOTHING = 0.1
WARMUP_EPOCHS   = 5
MIN_IMAGES      = 30

OUTPUT_DIR     = "/content/drive/MyDrive/VanaVaidhya_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)
BEST_CKPT      = f"{OUTPUT_DIR}/best_90.weights.h5"
TFLITE_PATH    = f"{OUTPUT_DIR}/plant_model.tflite"
LABEL_MAP_PATH = f"{OUTPUT_DIR}/label_map.json"

print(f"Batch size : {BATCH_SIZE}")
print(f"Image size : {IMG_SIZE}x{IMG_SIZE}")
print(f"Output dir : {OUTPUT_DIR}")

✅ GPU ready: /physical_device:GPU:0
✅ Mixed precision float16 enabled
Batch size : 128
Image size : 260x260
Output dir : /content/drive/MyDrive/VanaVaidhya_model


In [10]:
# CELL 7 — Collect Images & Splits
from pathlib import Path
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split

class_to_paths = defaultdict(list)
for root, mapping in [(Path(LEAF_DS), LEAF_DATASET_MAP), (Path(PLANT_DS), PLANT_DATASET_MAP)]:
    if not root.exists():
        print(f"⚠️  Not found: {root}"); continue
    for folder in root.iterdir():
        if not folder.is_dir(): continue
        cls = mapping.get(folder.name, "SKIP")
        if cls == "SKIP": continue
        imgs = list(folder.glob("**/*.jpg")) + list(folder.glob("**/*.jpeg")) + list(folder.glob("**/*.png"))
        class_to_paths[cls].extend(imgs)

valid       = {k: v for k, v in class_to_paths.items() if len(v) >= MIN_IMAGES}
class_names = sorted(valid.keys())
num_classes = len(class_names)
idx_of      = {n: i for i, n in enumerate(class_names)}
total       = sum(len(v) for v in valid.values())
print(f"✅ {num_classes} classes | {total} total images | avg {total//num_classes}/class")

all_p, all_l = [], []
for n in class_names:
    for p in valid[n]: all_p.append(str(p)); all_l.append(idx_of[n])

tr_p, tmp_p, tr_l, tmp_l = train_test_split(all_p, all_l, test_size=0.2, random_state=42, stratify=all_l)
va_p, te_p, va_l, te_l   = train_test_split(tmp_p, tmp_l, test_size=0.5, random_state=42, stratify=tmp_l)
print(f"Train: {len(tr_p)} | Val: {len(va_p)} | Test: {len(te_p)}")

cnt = Counter(tr_l)
class_weights = {i: len(tr_l)/(num_classes*cnt.get(i,1)) for i in range(num_classes)}

✅ 71 classes | 11534 total images | avg 162/class
Train: 9227 | Val: 1153 | Test: 1154


In [14]:
# CELL 8 — Data Pipeline (Augmentation + MixUp)
def load_img(path, label, aug=False):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    if aug:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, 0.25)
        img = tf.image.random_contrast(img, 0.7, 1.3)
        img = tf.image.random_saturation(img, 0.7, 1.3)
        img = tf.image.random_hue(img, 0.1)
        img = tf.image.random_crop(
            tf.image.resize_with_pad(img, IMG_SIZE+40, IMG_SIZE+40), [IMG_SIZE, IMG_SIZE, 3])
        img = tf.image.rot90(img, tf.random.uniform([], 0, 4, dtype=tf.int32))
    return tf.clip_by_value(img, 0.0, 1.0), label

def mixup_batch(images, labels):
    batch_size = tf.shape(images)[0]
    lam     = tf.maximum(tf.random.uniform([batch_size]), 1.0 - tf.random.uniform([batch_size]))
    lam_img = lam[:, tf.newaxis, tf.newaxis, tf.newaxis]
    idx     = tf.random.shuffle(tf.range(batch_size))
    mixed_x = lam_img * images + (1 - lam_img) * tf.gather(images, idx)
    oh_a    = tf.one_hot(labels, num_classes)
    oh_b    = tf.one_hot(tf.gather(labels, idx), num_classes)
    mixed_y = lam[:, tf.newaxis] * oh_a + (1 - lam[:, tf.newaxis]) * oh_b
    return mixed_x, mixed_y

def make_ds(p, l, aug=False, shuf=False, mixup=False):
    ds = tf.data.Dataset.from_tensor_slices((p, l))
    if shuf: ds = ds.shuffle(len(p), reshuffle_each_iteration=True)
    ds = ds.map(lambda x, y: load_img(x, y, aug), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.apply(tf.data.experimental.ignore_errors())
    if mixup: ds = ds.map(mixup_batch, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds     = make_ds(tr_p, tr_l, aug=True,  shuf=True,  mixup=False)
train_ds_mix = make_ds(tr_p, tr_l, aug=True,  shuf=True,  mixup=True)
val_ds       = make_ds(va_p, va_l)
test_ds      = make_ds(te_p, te_l)
print("✅ Data pipelines ready")

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


✅ Data pipelines ready


In [11]:
# CELL 9 — Build EfficientNetV2S Model
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras import layers, Model

base = EfficientNetV2S(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet", pooling="avg")

def freeze_all(m):
    for l in m.layers: l.trainable = False

def unfreeze_top_n_conv(m, n):
    freeze_all(m)
    non_bn = [l for l in m.layers if not isinstance(l, tf.keras.layers.BatchNormalization)]
    for l in non_bn[-n:]: l.trainable = True
    u = sum(1 for l in m.layers if l.trainable)
    b = sum(1 for l in m.layers if isinstance(l, tf.keras.layers.BatchNormalization))
    print(f"  Conv unfrozen: {u}  |  BN always frozen: {b}")

freeze_all(base)
inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x   = base(inp)
x   = layers.Dense(512, activation="relu")(x)
x   = layers.BatchNormalization()(x)
x   = layers.Dropout(0.4)(x)
out = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)
model = Model(inp, out)
print(f"✅ Model built | Params: {model.count_params():,} | Classes: {num_classes}")

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
✅ Model built | Params: 21,025,703 | Classes: 71


In [12]:
# CELL 10 — Callbacks (Cosine LR Warmup)
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def cosine_warmup(ep, lr, total_ep, base_lr, warmup_ep):
    if ep < warmup_ep:
        return float(base_lr * (ep + 1) / warmup_ep)
    return float(base_lr * 0.5 * (1 + np.cos(np.pi * (ep - warmup_ep) / (total_ep - warmup_ep))))

def make_cbs(phase, total_ep, base_lr):
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(
            BEST_CKPT, save_weights_only=True, save_best_only=True,
            monitor="val_accuracy", verbose=1),
        EarlyStopping(patience=15, restore_best_weights=True, monitor="val_accuracy", verbose=1),
    ]
    if phase in (1, 2):
        cbs.append(tf.keras.callbacks.LearningRateScheduler(
            lambda ep, lr: cosine_warmup(ep, lr, total_ep, base_lr, WARMUP_EPOCHS), verbose=0))
    else:
        cbs.append(ReduceLROnPlateau(monitor="val_accuracy", factor=0.5, patience=6, min_lr=1e-8, verbose=1))
    return cbs

print("✅ Callbacks configured")

✅ Callbacks configured


In [16]:
# CELL 11 — PHASE 1: Head Only (~20 min on T4)
# Expected: 50-65% val accuracy
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR1),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")],
)
print(f"\n{'='*60}")
print(f"  PHASE 1 — Head only | Epochs: {PHASE1_EPOCHS} | LR: {LR1} | Batch: {BATCH_SIZE}")
print(f"  Cosine LR with {WARMUP_EPOCHS}-epoch warmup")
print(f"{'='*60}\n")

h1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    class_weight=class_weights,
    callbacks=make_cbs(1, PHASE1_EPOCHS, LR1),
)
print(f"\n✅ Phase 1 done | Best val_acc: {max(h1.history['val_accuracy'])*100:.1f}%")


  PHASE 1 — Head only | Epochs: 30 | LR: 0.0003 | Batch: 128
  Cosine LR with 5-epoch warmup

Epoch 1/30
     73/Unknown 271s 3s/step - accuracy: 0.0210 - loss: 4.4356 - top3: 0.0570

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 1: val_accuracy improved from None to 0.03209, saving model to /content/drive/MyDrive/VanaVaidhya_model/best_90.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/VanaVaidhya_model/best_90.weights.h5
73/73 ━━━━━━━━━━━━━━━━━━━━ 339s 4s/step - accuracy: 0.0218 - loss: 4.4449 - top3: 0.0597 - val_accuracy: 0.0321 - val_loss: 4.2270 - val_top3: 0.0772 - learning_rate: 6.0000e-05
Epoch 2/30
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.0197 - loss: 4.3204 - top3: 0.0628
Epoch 2: val_accuracy did not improve from 0.03209
73/73 ━━━━━━━━━━━━━━━━━━━━ 203s 3s/step - accuracy: 0.0234 - loss: 4.3478 - top3: 0.0686 - val_accuracy: 0.0321 - val_loss: 4.2015 - val_top3: 0.0954 - learning_rate: 1.2000e-04
Epoch 3/30
72/73 ━━━━━━━━━━━━━━━━━━━━ 1s 2s/step - accuracy: 0.0270 - loss: 4.3092 - top3: 0.0755
Epoch 3: val_accuracy improved from 0.03209 to 0.03903, saving model to /content/drive/MyDrive/VanaVaidhya_model/best_90.weights.h5

Epoch 3: finished saving model to /cont

In [ ]:
# CELL 12 — PHASE 2: Top-100 Conv + MixUp + Label Smoothing (~20 min)
# Expected: 80-87% val accuracy
unfreeze_top_n_conv(base, 100)
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR2, clipnorm=1.0),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3")],
)
print(f"\n{'='*60}")
print(f"  PHASE 2 — Top-100 conv + MixUp + Label Smoothing={LABEL_SMOOTHING}")
print(f"  Epochs: {PHASE2_EPOCHS} | LR: {LR2}")
print(f"{'='*60}\n")

h2 = model.fit(
    train_ds_mix, validation_data=val_ds,
    epochs=PHASE1_EPOCHS + PHASE2_EPOCHS,
    initial_epoch=PHASE1_EPOCHS,
    callbacks=make_cbs(2, PHASE1_EPOCHS + PHASE2_EPOCHS, LR2),
)
print(f"\n✅ Phase 2 done | Best val_acc: {max(h2.history['val_accuracy'])*100:.1f}%")

In [ ]:
# CELL 13 — PHASE 3: ALL Conv Layers, Tiny LR (~15 min)
# Expected: 88-93% val accuracy
all_non_bn = [l for l in base.layers if not isinstance(l, tf.keras.layers.BatchNormalization)]
unfreeze_top_n_conv(base, len(all_non_bn))
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR3, clipnorm=1.0),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3")],
)
total_eps = PHASE1_EPOCHS + PHASE2_EPOCHS + PHASE3_EPOCHS
print(f"\n{'='*60}")
print(f"  PHASE 3 — ALL conv unlocked | LR: {LR3} | Epochs: {PHASE3_EPOCHS}")
print(f"{'='*60}\n")

h3 = model.fit(
    train_ds_mix, validation_data=val_ds,
    epochs=total_eps,
    initial_epoch=PHASE1_EPOCHS + PHASE2_EPOCHS,
    callbacks=make_cbs(3, total_eps, LR3),
)
print(f"\n✅ Phase 3 done | Best val_acc: {max(h3.history['val_accuracy'])*100:.1f}%")

In [ ]:
# CELL 14 — Evaluate + Test-Time Augmentation (TTA)
model.load_weights(BEST_CKPT)
print("✅ Loaded best weights\n")

freeze_all(base)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")])

loss, acc, top3 = model.evaluate(test_ds, verbose=1)
print(f"\n  Standard Accuracy : {acc*100:.2f}%")
print(f"  Top-3 Accuracy    : {top3*100:.2f}%")

# TTA: 8 augmented passes → average → free +3-5%
print("\n🔮 Running TTA (8 passes)...")
tta_preds = []
for i in range(8):
    preds = model.predict(make_ds(te_p, te_l, aug=True), verbose=0)
    tta_preds.append(preds)
    print(f"  Pass {i+1}/8 done")

avg_preds  = np.mean(tta_preds, axis=0)
tta_labels = np.array(te_l)
tta_acc    = np.mean(np.argmax(avg_preds, axis=1) == tta_labels)
tta_top3   = np.mean(np.any(
    np.argsort(avg_preds, axis=1)[:, -3:] == tta_labels[:, np.newaxis], axis=1))

print(f"\n{'='*60}")
print(f"  ✅ Standard Accuracy : {acc*100:.2f}%")
print(f"  🔮 TTA Accuracy      : {tta_acc*100:.2f}%  (+{(tta_acc-acc)*100:.1f}%)")
print(f"  Top-3 (TTA)         : {tta_top3*100:.2f}%")
print(f"{'='*60}")

In [ ]:
# CELL 15 — Save Label Map
import json
DISPLAY_NAMES = {
    "aloe_vera": "Aloe Vera", "amla": "Amla (Indian Gooseberry)",
    "amruthaballi": "Amruthaballi (Giloy)", "arali": "Arali (Nerium) ⚠️",
    "ashoka": "Ashoka Tree", "ashwagandha": "Ashwagandha",
    "asthma_weed": "Asthma Weed", "bael": "Bael (Bel)",
    "badipala": "Badipala", "balloon_vine": "Balloon Vine",
    "bamboo": "Bamboo", "betel": "Betel Leaf (Paan)",
    "betel_nut": "Betel Nut (Supari)", "brahmi": "Brahmi",
    "bringaraja": "Bhringraj", "camphor": "Camphor",
    "caricature": "Caricature Plant", "castor": "Castor (Arandi)",
    "catharanthus": "Catharanthus (Sadabahar)", "chakte": "Chakte",
    "citron_lime": "Citron Lime", "common_rue": "Common Rue",
    "coriander": "Coriander (Dhaniya)", "curry_leaf": "Curry Leaf",
    "custard_apple": "Custard Apple", "doddpathre": "Doddpathre",
    "ekka": "Ekka", "eucalyptus": "Eucalyptus",
    "ganigale": "Ganigale", "ganike": "Ganike",
    "gasagase": "Gasagase (Poppy)", "geranium": "Geranium",
    "ginger": "Ginger (Adrak)", "globe_amaranth": "Globe Amaranth",
    "guava": "Guava (Amrood)", "harsingar": "Harsingar (Parijat)",
    "henna": "Henna (Mehendi)", "hibiscus": "Hibiscus (Gudhal)",
    "honge": "Honge (Pongamia)", "insulin_plant": "Insulin Plant",
    "jackfruit": "Jackfruit", "jasmine": "Jasmine (Chameli)",
    "kamakasturi": "Kamakasturi", "kambajala": "Kambajala",
    "kasambruga": "Kasambruga", "kepala": "Kepala",
    "lantana": "Lantana ⚠️ (Toxic)", "lemon": "Lemon (Nimbu)",
    "lemongrass": "Lemongrass", "malabar_nut": "Malabar Nut",
    "mango": "Mango (Aam)", "marigold": "Marigold (Genda)",
    "mint": "Mint (Pudina)", "moringa": "Drumstick (Moringa)",
    "neem": "Neem", "nelavembu": "Nelavembu (Kalmegh)",
    "nerale": "Indian Blackberry (Jamun)", "noni": "Noni",
    "padri": "Padri", "papaya": "Papaya", "pepper": "Black Pepper",
    "pomegranate": "Pomegranate (Anar)", "raktachandini": "Raktachandini",
    "rose": "Rose (Gulab)", "sampige": "Champak (Sampige)",
    "sapota": "Sapota (Chikoo)", "tamarind": "Tamarind (Imli)",
    "tecoma": "Tecoma", "thumbe": "Thumbe", "tulsi": "Tulsi (Holy Basil)",
    "turmeric": "Turmeric (Haldi)", "wood_sorel": "Wood Sorrel",
}
label_map = {
    str(i): {
        "class_name": n,
        "display_name": DISPLAY_NAMES.get(n, n.replace("_", " ").title()),
        "is_toxic": n in TOXIC_PLANTS,
        "needs_caution": n in CAUTION_PLANTS,
        "ayush_recognized": n in AYUSH_PLANTS,
    }
    for i, n in enumerate(class_names)
}
with open(LABEL_MAP_PATH, "w") as f:
    json.dump(label_map, f, indent=2)
print(f"✅ Label map saved → {LABEL_MAP_PATH}  ({num_classes} classes)")

In [ ]:
# CELL 16 — Export TFLite (INT8 quantized, ~20MB)
import random as py_random
from PIL import Image as PILImage

tf.keras.mixed_precision.set_global_policy("float32")
freeze_all(base)

# Rebuild in float32 for TFLite conversion
inp32  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x32    = base(inp32)
x32    = layers.Dense(512, activation="relu")(x32)
x32    = layers.BatchNormalization()(x32)
x32    = layers.Dropout(0.4)(x32)
out32  = layers.Dense(num_classes, activation="softmax", dtype="float32")(x32)
model32 = Model(inp32, out32)
model32.load_weights(BEST_CKPT)
print("✅ Float32 model loaded")

def rep_ds():
    for p in py_random.sample(tr_p, min(200, len(tr_p))):
        try:
            img = PILImage.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
            yield [np.array(img, dtype=np.float32)[np.newaxis] / 255.0]
        except Exception:
            continue

print("🔄 Converting to TFLite (INT8 quantized)...")
conv = tf.lite.TFLiteConverter.from_keras_model(model32)
conv.optimizations             = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset    = rep_ds
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8, tf.lite.OpsSet.SELECT_TF_OPS]
conv.inference_input_type      = tf.float32
conv.inference_output_type     = tf.float32

try:
    tflite_bytes = conv.convert()
except Exception as e:
    print(f"⚠️  INT8 failed ({e}), using float16...")
    conv2 = tf.lite.TFLiteConverter.from_keras_model(model32)
    conv2.optimizations = [tf.lite.Optimize.DEFAULT]
    conv2.target_spec.supported_types = [tf.float16]
    tflite_bytes = conv2.convert()

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_bytes)
mb = os.path.getsize(TFLITE_PATH) / 1024 / 1024
print(f"\n✅ TFLite saved → {TFLITE_PATH}  ({mb:.1f} MB)")
print(f"""
╔══════════════════════════════════════════════════════╗
║  ✅  TRAINING COMPLETE!                              ║
║                                                      ║
║  Standard Accuracy : {acc*100:.1f}%                         ║
║  TTA Accuracy      : {tta_acc*100:.1f}%  ← used in the app  ║
║  Model size        : {mb:.1f} MB                           ║
║                                                      ║
║  Files saved to Google Drive:                        ║
║    VanaVaidhya_model/plant_model.tflite              ║
║    VanaVaidhya_model/label_map.json                  ║
║                                                      ║
║  Copy both to: VanaVaidhya/assets/model/             ║
╚══════════════════════════════════════════════════════╝
""")